# AI2002 - Group 5 - Review Chatbot for Hotel Question Answering

## Đề tài: Xây dựng Chatbot thông minh hỗ trợ trả lời câu hỏi và tư vấn khách sạn dựa trên tập dữ liệu đánh giá TripAdvisor (Hotel Review Question Answering)

## 0. Cài đặt môi trường (Environment Setup & Auto-install)

Với dự án nghiên cứu về **Review Chatbot for Hotel Question Answering**, hệ thống được thiết lập cơ chế **tự động kiểm tra và cài đặt thư viện** từ file `requirements.txt` vào môi trường ảo (`.venv`):

1. **Tự động nhận diện môi trường ảo**: Cài đặt trực tiếp vào Python Kernel (`sys.executable`) đang chạy trong notebook.
2. **Kiểm tra thông minh**: Chỉ cài đặt các gói còn thiếu từ `requirements.txt`, tránh tốn thời gian khi chạy lại notebook.
3. **Các thư viện chính**:
   - **Pandas / NumPy**: Xử lý dữ liệu bảng quy mô lớn và tính toán đại số ma trận.
   - **SQLite3**: Cơ sở dữ liệu quan hệ cục bộ lưu trữ dữ liệu có cấu trúc.
   - **Matplotlib / Seaborn**: Trực quan hóa dữ liệu EDA và biểu đồ phân tích.
   - **Scikit-learn / NLTK**: Xử lý ngôn ngữ tự nhiên (NLP), TF-IDF và thuật toán Cosine Similarity.
   - **Tqdm / Ipywidgets**: Thanh tiến trình hiển thị trực quan trong quá trình xử lý văn bản.


In [ ]:
import sys
import os
import subprocess
import re
import importlib.metadata

# ----------------------------------------------------
# TỰ ĐỘNG CÀI ĐẶT THƯ VIỆN TỪ REQUIREMENTS.TXT VÀO VENV
# ----------------------------------------------------
REQ_FILE = 'requirements.txt'

def auto_install_requirements(req_path=REQ_FILE):
    """
    Kiểm tra và tự động cài đặt các gói phụ thuộc từ requirements.txt
    vào chính môi trường ảo (venv) đang vận hành Notebook Kernel.
    """
    # Xác định đường dẫn file requirements.txt
    search_paths = [
        req_path,
        os.path.join(os.getcwd(), req_path),
        os.path.join(os.path.dirname(os.getcwd()), req_path),
        os.path.join('..', req_path)
    ]
    found_path = next((p for p in search_paths if os.path.exists(p)), None)

    if not found_path:
        print(f"⚠️ Cảnh báo: Không tìm thấy file '{req_path}'. Vui lòng kiểm tra lại thư mục.")
        return

    print(f"[1/2] Đang kiểm tra môi trường: {sys.executable}")
    print(f"      File cấu hình yêu cầu: {os.path.abspath(found_path)}\n")

    # Đọc và đối chiếu các gói cần thiết
    missing = []
    with open(found_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            pkg_name = re.split(r'[><=~!]', line)[0].strip()
            try:
                importlib.metadata.version(pkg_name)
            except importlib.metadata.PackageNotFoundError:
                missing.append(line)

    if missing:
        print(f"🔄 Phát hiện {len(missing)} thư viện chưa có trong môi trường:")
        for item in missing:
            print(f"   • {item}")
        print("\n⏳ Đang tự động cài đặt vào môi trường venv qua pip...")
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', found_path])
            print("\n✅ Đã cài đặt hoàn tất tất cả thư viện từ requirements.txt!")
        except subprocess.CalledProcessError as e:
            print(f"❌ Quá trình cài đặt gặp lỗi: {e}")
    else:
        print("✅ Tất cả thư viện trong requirements.txt đã được cài đặt đầy đủ trong môi trường ảo (.venv)!")

# Thực hiện kiểm tra và cài đặt
auto_install_requirements()


In [ ]:
import os
import re
import sqlite3
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import nltk

# Tự động tải các gói tài nguyên cần thiết cho NLTK (không làm gián đoạn nếu đã có)
for res in ['punkt', 'punkt_tab', 'stopwords']:
    try:
        nltk.download(res, quiet=True)
    except Exception:
        pass

# Hiển thị bảng phiên bản chi tiết các thư viện
print(f"{'Thư viện':<20} {'Phiên bản':>15}")
print("-" * 37)
print(f"{'Python':<20} {os.sys.version.split()[0]:>15}")
print(f"{'Pandas':<20} {pd.__version__:>15}")
print(f"{'NumPy':<20} {np.__version__:>15}")
print(f"{'Matplotlib':<20} {matplotlib.__version__:>15}")
print(f"{'Seaborn':<20} {sns.__version__:>15}")
print(f"{'Scikit-learn':<20} {sklearn.__version__:>15}")
print(f"{'NLTK':<20} {nltk.__version__:>15}")
print(f"{'SQLite3':<20} {sqlite3.sqlite_version:>15}")
print(f"{'Regex (re)':<20} {re.__version__ if hasattr(re, '__version__') else 'built-in':>15}")

# Khởi tạo thư mục và kết nối CSDL SQLite
db_dir = 'Data' if os.path.exists('Data') else 'Data (Dataset, Data Frame, Chart Images)'
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, 'chatbot.db')
conn = sqlite3.connect(db_path)

print(f"\n📁 CSDL SQLite đã sẵn sàng tại: {db_path}")
print("--- Môi trường cho Hotel Review Chatbot đã được khởi tạo thành công ---")


Thư viện                   Phiên bản
-------------------------------------
Python                       3.12.14
Pandas                         3.0.5
NumPy                          2.5.3
Matplotlib                    3.11.1
Seaborn                       0.13.2
Scikit-learn                   1.9.0
NLTK                          3.10.3
SQLite3                       3.53.4
Regex (re)                     2.2.1

--- Môi trường cho Hotel Review Chatbot đã được khởi tạo thành công ---


## 1. Nhập và Kiểm tra Dataset (Dataset Loading & Exploration)

Trong phần này, nhóm sẽ thực hiện các bước chuẩn bị và khảo sát ban đầu:
1. **Tải tập dữ liệu nghiên cứu**: Đọc tập dữ liệu đánh giá khách sạn từ file CSV (`tripadvisor_review_hotel_dataset.csv`). Chương trình tự động kiểm tra sự tồn tại của file ở thư mục hiện tại, thư mục lưu trữ dữ liệu `Data (Dataset, Data Frame, Chart Images)/` hoặc trên Google Drive.
2. **Chuyển đổi vào Pandas DataFrame**: Sử dụng bộ mã hóa UTF-8 để đảm bảo hiển thị chuẩn xác tiếng Việt và các ngôn ngữ quốc tế.
3. **Kiểm tra cấu trúc và các trường thông tin quan trọng**:
   - **Thông tin cơ sở lưu trú**: `hotel_name`, `hotel_province`, `hotel_address`, `hotel_star`.
   - **Điểm đánh giá và các khía cạnh dịch vụ**: `normalized_score`, `Value`, `Rooms`, `Location`, `Cleanliness`, `Service`, `Sleep_Quality`.
   - **Dữ liệu văn bản phục vụ Chatbot Hỏi Đáp (QA)**: `normalized_title` (tiêu đề đánh giá), `normalized_content` (nội dung đánh giá chi tiết), `Word_count`, `language_code`, `language`.
   - **Bối cảnh chuyến đi & Thời gian**: `trip_type`, `Date`, `month`, `year`.

In [ ]:
import os
import pandas as pd

# Tạo thư mục lưu trữ biểu đồ trực quan hóa nếu chưa tồn tại
os.makedirs('charts_img', exist_ok=True)

# Danh sách các đường dẫn ứng viên của dataset (hỗ trợ cả Local và Google Colab)
candidate_paths = [
    'Data/tripadvisor_review_hotel_dataset.csv',
    'Data (Dataset, Data Frame, Chart Images)/tripadvisor_review_hotel_dataset.csv',
    'tripadvisor_review_hotel_dataset.csv',
    '/content/drive/MyDrive/Data (Dataset, Data Frame, Chart Images)/tripadvisor_review_hotel_dataset.csv',
    '/content/drive/MyDrive/tripadvisor_review_hotel_dataset.csv',
    '/content/tripadvisor_review_hotel_dataset.csv'
]

file_path = None
for path in candidate_paths:
    if os.path.exists(path):
        file_path = path
        print(f"Đã tìm thấy dataset tại: {file_path}")
        break

if file_path is None:
    file_path = candidate_paths[0]
    print(f"Cảnh báo: Chưa tìm thấy file dataset. Hãy đảm bảo file '{os.path.basename(file_path)}' đã được tải lên.")

# Tiến hành đọc file CSV
try:
    df = pd.read_csv(file_path, encoding='utf-8', low_memory=False)
    print(f"Kích thước tập dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột")
    print("\nCác cột dữ liệu phục vụ nghiên cứu:")
    print(list(df.columns))
    
    # Hiển thị 5 dòng đầu tiên
    print("\nXem trước 5 dòng đầu tiên:")
    display(df.head())
except Exception as e:
    print(f"Lỗi khi đọc file CSV: {e}")
